# Index · Unit 3 — Knowledge & grounding (RAG)

Same shape as Units 1–2. For every component in **Section 4 of `index.md`**, two cells:

- **From scratch** — the mechanism by hand, no framework. Runs on plain Python /
  a scripted *fake model* (no API key needed) so you see every moving part.
- **SOTA** — how it's actually done today (OpenAI embeddings, re-rankers,
  structured outputs). Needs `OPENAI_API_KEY`.

Components: **1** embeddings & vector DBs · **2** chunking / indexing /
re-ranking · **3** RAG (retrieve → augment → generate) · **4** knowledge graphs
(GraphRAG) · **5** hallucination mitigation (grounding + citations).

**Intuition for the whole unit:** a model only knows what it was trained on plus
what's in the window. **Grounding** = fetch the right *external* text at query
time and put it in the window, so answers rest on real sources instead of the
model's memory. Everything here serves that one loop: *find relevant text →
hand it to the model → make it answer only from that text, with citations.*

> Order note: `index.md` lists RAG first, but RAG is the *assembly* of the other
> parts — so we build embeddings → retrieval pipeline → RAG, then graphs and
> grounding. Same five components, bottom-up.

## Setup

In [ ]:
import os, re, json, math

MODEL = "gpt-4o-mini"
EMBED = "text-embedding-3-small"

print("✅ key set — SOTA cells will run" if os.environ.get("OPENAI_API_KEY")
      else "⚠️  no OPENAI_API_KEY — from-scratch cells still run; SOTA cells need a key")

def chat(messages, model=MODEL, **kwargs):
    from openai import OpenAI
    return OpenAI().chat.completions.create(
        model=model, messages=messages, **kwargs).choices[0].message

def embed(texts):
    from openai import OpenAI
    r = OpenAI().embeddings.create(model=EMBED, input=texts)
    return [d.embedding for d in r.data]

def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    na, nb = math.sqrt(sum(x*x for x in a)), math.sqrt(sum(y*y for y in b))
    return dot / (na*nb + 1e-9)

# A tiny corpus we reuse across the unit.
DOCS = [
    "The Eiffel Tower is in Paris and is 330 meters tall.",
    "Paris is the capital of France.",
    "The Great Wall of China is over 20,000 km long.",
    "Mount Everest is the tallest mountain at 8,849 meters.",
    "France borders Germany, Spain, Italy, and Belgium.",
]

## 1. Embeddings & vector databases

An **embedding** turns text into a vector of numbers so that *meaning* becomes
*geometry*: similar texts land close together. A **vector DB** stores those
vectors and answers "which stored texts are nearest to this query vector?" —
that's semantic search, the backbone of RAG.

**From scratch:** a bag-of-words embedding (count shared words) over a fixed
vocabulary + a list-backed store with cosine search. Crude, but it's a real
vector index.

In [ ]:
# From scratch — bag-of-words vectors + a list-backed vector store.
def tokenize(t): return re.findall(r"[a-z]+", t.lower())
VOCAB = sorted({w for d in DOCS for w in tokenize(d)})

def bow(text):
    counts = {w: 0 for w in VOCAB}
    for w in tokenize(text):
        if w in counts: counts[w] += 1
    return [counts[w] for w in VOCAB]

class VectorDB:
    def __init__(self, embed_fn): self.embed_fn, self.texts, self.vecs = embed_fn, [], []
    def add(self, texts):
        self.texts += texts; self.vecs += [self.embed_fn(t) for t in texts]
    def search(self, query, k=2):
        qv = self.embed_fn(query)
        scored = sorted(zip(self.texts, self.vecs), key=lambda tv: cosine(qv, tv[1]), reverse=True)
        return [t for t, _ in scored[:k]]

db = VectorDB(bow); db.add(DOCS)
print(db.search("How tall is the tower in France?"))

In [ ]:
# SOTA — same store, real neural embeddings (meaning, not word overlap).
sota_db = VectorDB(lambda t: embed([t])[0]); sota_db.add(DOCS)
# "French capital" shares no content words with "Paris is the capital of France"
# beyond 'capital' — neural embeddings still rank it correctly.
print(sota_db.search("What's the French capital?"))
print(sota_db.search("elevation of the world's highest peak"))

## 2. Chunking, indexing & re-ranking — the retrieval pipeline

Real documents are too long to embed whole, so you **chunk** them into passages,
**index** the chunks (embed + store), retrieve a cheap top-K, then **re-rank**
that shortlist with a more precise (slower) scorer to get the final top-k. The
two-stage *retrieve-then-rerank* pattern is standard: cast a wide cheap net,
then sort precisely.

**From scratch:** fixed-size word chunks, index into our VectorDB, retrieve
top-K by cosine, re-rank by exact keyword overlap.

In [ ]:
# From scratch — chunk -> index -> retrieve K -> re-rank to k.
DOCUMENT = ("Paris is the capital of France. The Eiffel Tower, built in 1889, "
            "stands 330 meters tall on the Champ de Mars. It was the tallest "
            "man-made structure in the world for 41 years until the Chrysler "
            "Building was finished in New York in 1930.")

def chunk(text, size=8):
    words = text.split()
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size)]

chunks = chunk(DOCUMENT)
index = VectorDB(bow); index.add(chunks)

def retrieve_rerank(query, K=4, k=2):
    shortlist = index.search(query, k=K)                 # stage 1: cheap recall
    q = set(tokenize(query))
    reranked = sorted(shortlist, key=lambda c: len(q & set(tokenize(c))), reverse=True)
    return reranked[:k]                                  # stage 2: precise rank

print("chunks:", len(chunks))
for c in retrieve_rerank("How tall is the Eiffel Tower?"): print(" •", c)

In [ ]:
# SOTA — neural retrieve, then LLM re-rank (stand-in for a cross-encoder re-ranker).
nindex = VectorDB(lambda t: embed([t])[0]); nindex.add(chunks)

def sota_retrieve_rerank(query, K=4, k=2):
    shortlist = nindex.search(query, k=K)
    scored = chat([
        {"role": "system", "content": "Score each passage 0-10 for how well it answers the "
         "question. Reply JSON: {\"scores\":[...]} in passage order."},
        {"role": "user", "content": json.dumps({"question": query, "passages": shortlist})},
    ], response_format={"type": "json_object"})
    order = sorted(range(len(shortlist)),
                   key=lambda i: json.loads(scored.content)["scores"][i], reverse=True)
    return [shortlist[i] for i in order[:k]]

print(sota_retrieve_rerank("How tall is the Eiffel Tower?"))

## 3. RAG — retrieve → augment → generate

RAG is just the assembly: **retrieve** relevant chunks, **augment** the prompt by
stuffing them in as context, then **generate** an answer grounded in that
context. The model's own parametric knowledge is backup; the retrieved text is
the source of truth.

**From scratch:** retrieve with our index, build a context block, and a *fake
model* that answers strictly from the provided context (so you see the data
flow without a key).

In [ ]:
# From scratch — retrieve, stuff context, generate (fake grounded model).
def fake_grounded(query, context):
    # a real model reasons; this stand-in just finds the sentence with a number.
    for line in context.split("\n"):
        if re.search(r"\d", line) and any(w in line.lower() for w in tokenize(query)):
            return f"Based on the context: {line.strip()}"
    return "I don't know from the provided context."

def rag(query, generate):
    hits = retrieve_rerank(query, K=4, k=2)
    context = "\n".join(f"- {h}" for h in hits)
    print("retrieved context:\n" + context)
    return generate(query, context)

print("\nanswer:", rag("How tall is the Eiffel Tower?", fake_grounded))

In [ ]:
# SOTA — real retrieval + a model told to answer only from context.
def sota_rag(query):
    hits = sota_retrieve_rerank(query, K=4, k=2)
    context = "\n".join(f"- {h}" for h in hits)
    m = chat([
        {"role": "system", "content": "Answer the question using ONLY the context. "
         "If the answer isn't there, say you don't know."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ])
    return m.content

print(sota_rag("How tall is the Eiffel Tower?"))

## 4. Knowledge graphs — GraphRAG

Vector search finds *similar text*; it's weak at *multi-hop relational* questions
("which countries border the country whose capital is Paris?"). A **knowledge
graph** stores facts as `(subject, relation, object)` **triples** you can
traverse. **GraphRAG** retrieves by walking these edges instead of (or alongside)
vector similarity.

**From scratch:** a triple store + a tiny traversal query.

In [ ]:
# From scratch — a triple store you can traverse.
TRIPLES = [
    ("Paris", "capital_of", "France"),
    ("Eiffel Tower", "located_in", "Paris"),
    ("France", "borders", "Germany"),
    ("France", "borders", "Spain"),
    ("Germany", "capital_of_rev", "Berlin"),
]

def neighbors(entity, relation):
    return [o for s, r, o in TRIPLES if s == entity and r == relation]

def multi_hop(start, rel1, rel2):
    # e.g. start=Eiffel Tower, located_in -> Paris, capital_of -> France
    out = []
    for mid in neighbors(start, rel1):
        out += neighbors(mid, rel2)
    return out

print("Eiffel Tower -> located_in -> capital_of:",
      multi_hop("Eiffel Tower", "located_in", "capital_of"))
print("France borders:", neighbors("France", "borders"))

In [ ]:
# SOTA — let the model EXTRACT triples from free text, then query the graph.
def extract_triples(text):
    m = chat([
        {"role": "system", "content": "Extract facts as triples. Reply JSON: "
         "{\"triples\": [[\"subject\",\"relation\",\"object\"], ...]}"},
        {"role": "user", "content": text},
    ], response_format={"type": "json_object"})
    return [tuple(t) for t in json.loads(m.content)["triples"]]

kg = extract_triples("Ada Lovelace worked with Charles Babbage on the Analytical Engine. "
                     "She is regarded as the first computer programmer.")
for t in kg: print(t)

## 5. Hallucination mitigation — grounding + citations

The point of all the above is **trust**: the answer must come from the retrieved
sources, and must say *which* source — a **citation** — so it's verifiable. And
when the sources don't contain the answer, the system must **abstain** rather
than invent one.

**From scratch:** give each chunk an ID, answer only from matched chunks, attach
the IDs, and refuse when nothing matches.

In [ ]:
# From scratch — grounded answer with citations + abstention.
CORPUS = {f"D{i}": d for i, d in enumerate(DOCS)}

def grounded_answer(query):
    q = set(tokenize(query))
    matches = [(cid, text) for cid, text in CORPUS.items()
               if len(q & set(tokenize(text))) >= 2]           # needs real overlap
    if not matches:
        return "I don't have a source for that."               # abstain, don't guess
    cid, text = max(matches, key=lambda m: len(q & set(tokenize(m[1]))))
    return f"{text}  [source: {cid}]"

print(grounded_answer("How tall is the Eiffel Tower?"))
print(grounded_answer("Who won the 2026 World Cup?"))     # not in corpus -> abstains

In [ ]:
# SOTA — model cites chunk IDs; then we VERIFY each citation actually supports it.
def cited_rag(query):
    hits = {f"D{i}": t for i, t in enumerate(sota_retrieve_rerank(query, K=4, k=3))}
    m = chat([
        {"role": "system", "content": "Answer ONLY from the sources. Cite the source id(s) "
         "in [brackets]. If unsupported, say you don't know."},
        {"role": "user", "content": f"Sources: {json.dumps(hits)}\n\nQuestion: {query}"},
    ])
    answer = m.content
    cited = re.findall(r"D\d", answer)
    ok = all(cid in hits for cid in cited) and cited      # every citation is a real source
    return {"answer": answer, "citations": cited, "citations_valid": bool(ok)}

print(json.dumps(cited_rag("How tall is the Eiffel Tower?"), indent=2))

## Recap

Grounding is one loop — *find relevant text → put it in the window → answer only
from it, with citations* — assembled from five parts:

| component | mechanism (from scratch) | SOTA packaging |
|---|---|---|
| embeddings & vector DB | bag-of-words vectors + cosine search | neural embeddings (`text-embedding-3`) |
| chunking / rerank | word chunks, retrieve-K then keyword rerank | neural recall + cross-encoder / LLM rerank |
| RAG | retrieve → stuff context → grounded answer | same loop with a real model, "use only context" |
| knowledge graph | `(s,r,o)` triples + traversal | LLM extracts triples, query the graph (GraphRAG) |
| grounding + citations | match chunks, attach IDs, abstain | model cites IDs, then verify each citation |

The through-line from Units 1–2 holds: each piece is a few lines of plain Python
(a cosine, a chunker, a dict of triples), and SOTA swaps in native model features
(neural embeddings, re-rankers, structured outputs). Frameworks (LlamaIndex,
LangChain, Haystack) package exactly this pipeline.

Next in `index.md`: **Unit 4 — Multi-agent systems** (Section 5).